In [20]:
from qiskit_aer import AerSimulator
from qiskit.circuit import *
from qiskit import QuantumCircuit, transpile

In [21]:
n = 3
while True:
    s = input("Digite uma string binária s de n bits: ").strip()
    if len(s) == n and all(bit in ('0', '1') for bit in s) and s != '0' * n:
        break
    print("Entrada inválida! Digite exatamente n bits ('0' ou '1'), diferente de '000'.")

In [22]:
def oraculo(s: str) -> QuantumCircuit:
    n = len(s)
    orac = QuantumCircuit(2*n, name="U_f")

    # inverte para little-endian do Qiskit
    s_rev = s[::-1]

    for i in range(n):
        orac.cx(i, n+i)

    if '1' in s_rev:
        indice = s_rev.index('1')
        for i in range(n):
            if s_rev[i] == '1':
                orac.cx(indice, n+i)

    return orac

circuito_oraculo = oraculo(s=s)
print("Circuito do Oráculo U_f:")
print(circuito_oraculo.draw('text'))

Circuito do Oráculo U_f:
                         
q_0: ──■─────────────────
       │                 
q_1: ──┼────■────────────
       │    │            
q_2: ──┼────┼────■────■──
     ┌─┴─┐  │    │    │  
q_3: ┤ X ├──┼────┼────┼──
     └───┘┌─┴─┐  │    │  
q_4: ─────┤ X ├──┼────┼──
          └───┘┌─┴─┐┌─┴─┐
q_5: ──────────┤ X ├┤ X ├
               └───┘└───┘


In [ ]:
n = len(s)

qr1 = QuantumRegister(n, name='x_reg')
qr2 = QuantumRegister(n, name='fx_reg')
cr1 = ClassicalRegister(n, name='c_y')
cr2 = ClassicalRegister(n, name='c_fx')

circuito_quantico = QuantumCircuit(qr1, qr2, cr1, cr2)

circuito_quantico.h(qr1)
circuito_quantico.barrier()

circuito_quantico.compose(circuito_oraculo, qubits=[*qr1, *qr2], inplace=True)
circuito_quantico.barrier()

circuito_quantico.measure(qr2, cr2)
circuito_quantico.barrier()

circuito_quantico.h(qr1)
circuito_quantico.barrier()

circuito_quantico.measure(qr1, cr1)

print(circuito_quantico.draw('text'))

def adicionar_vetor_base(basis: list, y_str: str) -> bool:
    """Insere y_str na base em RREF sobre GF(2) se for linearmente independente."""
    if y_str == '0' * len(y_str):
        return False

    vec = [int(bit) for bit in y_str]
    for pivot_col, b_vec in basis:
        if vec[pivot_col] == 1:
            vec = [v ^ b for v, b in zip(vec, b_vec)]

    if 1 not in vec:
        return False

    pivot_col = vec.index(1)
    for idx in range(len(basis)):
        p_col, b_vec = basis[idx]
        if b_vec[pivot_col] == 1:
            basis[idx] = (p_col, [b ^ v for b, v in zip(b_vec, vec)])
    basis.append((pivot_col, vec))
    return True

simulador = AerSimulator()
circuito_quant_compilado = transpile(circuito_quantico, simulador)

basis = []
y_counts = {}
tentativa = 0

while len(basis) < n - 1:
    tentativa += 1
    resultado = simulador.run(circuito_quant_compilado, shots=1).result()
    saida = next(iter(resultado.get_counts()))
    fx_medido, y_medido = saida.split(' ')
    y_counts[y_medido] = y_counts.get(y_medido, 0) + 1

    produto_interno = sum(int(b1) * int(b2) for b1, b2 in zip(y_medido, s)) % 2
    adicionado = adicionar_vetor_base(basis, y_medido)
    status = "adicionado à base (LI)" if adicionado else "descartado (nulo ou LD)"
    print(f"  Shot {tentativa:2d}: f(x) = {fx_medido} | y = {y_medido} | y · s mod 2 = {produto_interno} -> {status}")

print(f"\nBase completa com {n - 1} vetores LI após {tentativa} execução(ões).")

          ┌───┐ ░                      ░           ░ ┌───┐ ░ ┌─┐      
 x_reg_0: ┤ H ├─░───■──────────────────░───────────░─┤ H ├─░─┤M├──────
          ├───┤ ░   │                  ░           ░ ├───┤ ░ └╥┘┌─┐   
 x_reg_1: ┤ H ├─░───┼────■─────────────░───────────░─┤ H ├─░──╫─┤M├───
          ├───┤ ░   │    │             ░           ░ ├───┤ ░  ║ └╥┘┌─┐
 x_reg_2: ┤ H ├─░───┼────┼────■────■───░───────────░─┤ H ├─░──╫──╫─┤M├
          └───┘ ░ ┌─┴─┐  │    │    │   ░ ┌─┐       ░ └───┘ ░  ║  ║ └╥┘
fx_reg_0: ──────░─┤ X ├──┼────┼────┼───░─┤M├───────░───────░──╫──╫──╫─
                ░ └───┘┌─┴─┐  │    │   ░ └╥┘┌─┐    ░       ░  ║  ║  ║ 
fx_reg_1: ──────░──────┤ X ├──┼────┼───░──╫─┤M├────░───────░──╫──╫──╫─
                ░      └───┘┌─┴─┐┌─┴─┐ ░  ║ └╥┘┌─┐ ░       ░  ║  ║  ║ 
fx_reg_2: ──────░───────────┤ X ├┤ X ├─░──╫──╫─┤M├─░───────░──╫──╫──╫─
                ░           └───┘└───┘ ░  ║  ║ └╥┘ ░       ░  ║  ║  ║ 
   c_y: 3/════════════════════════════════╬══╬══╬═════════════╩══╩══╩═
      

In [ ]:
# Achar o s com os vetores ortogonais da base
def achar_string(basis: list, n: int) -> str:
    """Resolve y · s = 0 (mod 2) a partir da base em RREF com n - 1 vetores LI."""
    if len(basis) < n - 1:
        raise ValueError(f"Base insuficiente: {len(basis)} vetor(es) LI para n - 1 = {n - 1}.")

    pivot_cols = {p_col for p_col, _ in basis}
    free_col = [c for c in range(n) if c not in pivot_cols][0]

    s_encontrado = [0] * n
    s_encontrado[free_col] = 1
    for p_col, b_vec in basis:
        s_encontrado[p_col] = b_vec[free_col]

    return ''.join(str(bit) for bit in s_encontrado)

s_encontrado = achar_string(basis, n)
print("=" * 36)
print(f"String s fornecida  : {s}")
print(f"String s encontrada : {s_encontrado}")
print("=" * 36)
assert s_encontrado == s, "Divergência na recuperação da string!"